In [ ]:
# Fabric notebook source
# 01_Staging_Setup
# Run once before enabling orchestration. Safe to rerun because all DDL is guarded.

from pyspark.sql.functions import col

staging_schema = "LH_Staging"
staging_table = "LH_Staging.staging_rides"
watermark_table = "LH_Staging.load_control"
bronze_table = "LH_Bronze.bulk_rides"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {staging_schema}")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {watermark_table} (source_name STRING, last_processed_timestamp TIMESTAMP) USING DELTA""")

# Bootstrap historical bulk data only when staging has never been created.
if not spark.catalog.tableExists(staging_table):
    spark.sql(f"""
      CREATE TABLE {staging_table} USING DELTA AS
      SELECT b.*, CAST(NULL AS TIMESTAMP) AS bronze_ingestion_timestamp,
             CURRENT_TIMESTAMP() AS staging_load_timestamp
      FROM {bronze_table} b
    """)
    print("Historical staging bootstrap completed.")
else:
    print("Staging already exists. Bootstrap skipped.")

# Create an explicit watermark row without overwriting an existing watermark.
spark.sql(f"""
MERGE INTO {watermark_table} t
USING (SELECT 'EH_Bronze' AS source_name, CAST(NULL AS TIMESTAMP) AS last_processed_timestamp) s
ON t.source_name = s.source_name
WHEN NOT MATCHED THEN INSERT (source_name,last_processed_timestamp) VALUES (s.source_name,s.last_processed_timestamp)
""")
print("Staging setup complete.")